# Fraud Detection — Model Training
End-to-end pipeline: data loading → preprocessing → training → evaluation → saving the model.

## 1. Import Libraries

In [ ]:
import sys, os
sys.path.append(os.path.join("..", "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Libraries loaded.")

## 2. Load Dataset

In [ ]:
DATA_PATH = "../data/dataset.csv"

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print(df.info())
print("\nClass distribution:")
print(df["Class"].value_counts())

# Class imbalance visualisation
df["Class"].value_counts().plot(kind="bar", title="Class Distribution", color=["steelblue", "tomato"])
plt.xticks([0, 1], ["Legit", "Fraud"], rotation=0)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
TARGET = "Class"

X = df.drop(columns=[TARGET])
y = df[TARGET]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

## 5. Train Model

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced")
model.fit(X_train, y_train)
print("Training complete.")

## 6. Evaluate Model

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
MODEL_PATH = "../models/fraud_model.pkl"

joblib.dump({"model": model, "scaler": scaler}, MODEL_PATH)
print(f"Model saved → {MODEL_PATH}")